# Project D · Person 2 — Baseline forecaster and DCF

**The question this notebook serves:** can a language model that reads an industrial
firm's backlog commentary forecast FY2025 revenue better than a three-year average —
and how much does the difference matter for value?

This notebook does three things:

1. **The baseline.** The three-year-average growth and margin forecast for all 53 firms,
   plus a "last year" forecast as a second, tougher benchmark.
2. **The DCF.** One function that takes *any* year-1 growth and margin and returns an
   enterprise value. It is the measuring stick: it turns a forecast error into a
   valuation error.
3. **The sensitivity analysis.** Which assumption moves the value most — reported with
   the range each one was varied over — and whether the fixed assumptions can change
   which forecast wins.

The DCF is tested on dummy numbers (Part 2) before it touches the real data (Part 3).
Person 3's forecasts plug in at Part 4 without any other change.

## Part 0 · Data and assumptions

In [1]:
import os
import numpy as np
import pandas as pd

DATA = os.path.join("..", "data", "master.csv")
OUT = "output"
os.makedirs(OUT, exist_ok=True)

master = pd.read_csv(DATA)
mx = master.set_index("ticker")          # same table, indexed by ticker

print(f"firms          : {len(master)}")
print(f"target year    : {master['sample_year'].unique()}")
print(f"missing values : {master.isna().sum().sum()}")
print(f"duplicated     : {master['ticker'].duplicated().sum()}")
print(f"backlog firms  : {master['backlog_mentioned'].sum()} of {len(master)}")

firms          : 53
target year    : [2025]
missing values : 0
duplicated     : 0
backlog firms  : 38 of 53


Four firms have histories that are not like-for-like. They stay in the sample —
dropping them would be a choice nobody could see — but every summary below is also
reported without them.

In [2]:
FLAGGED = {
    "DD":  "FY22 revenue on a different basis to FY23-24 (restated); 'margin' is pre-tax",
    "FTV": "FY22 revenue on a different basis to FY23-24 (spin-off restatement)",
    "MMM": "FY23 margin of -43%: one-off litigation charges",
    "BA":  "operating losses FY22-24, then +34% revenue in FY25",
}
mx["flag"] = pd.Series(FLAGGED).reindex(mx.index).fillna("")

print(mx.loc[mx["flag"] != "", ["flag"]].to_string())

                                                                                flag
ticker                                                                              
MMM                                  FY23 margin of -43%: one-off litigation charges
BA                               operating losses FY22-24, then +34% revenue in FY25
DD      FY22 revenue on a different basis to FY23-24 (restated); 'margin' is pre-tax
FTV              FY22 revenue on a different basis to FY23-24 (spin-off restatement)


### The assumptions — every one written down

| Input | Value | Kind | Where it comes from |
| :--- | :--- | :--- | :--- |
| FY2024 revenue | per firm | filing | `revenue_fy_minus_1` (XBRL) |
| Year-1 growth, margin | per firm | **the forecast being tested** | baseline, last year, AI, or actual |
| Risk-free rate | 4.5% | assumed | 10-yr UST closed 4.49% on 7 Feb 2025, when the FY2024 10-Ks were filed |
| Equity risk premium | 5.0% | assumed | the course's ERP (Lecture 7); Damodaran's implied ERP was 4.33% on 1 Jan 2025 — run as a sensitivity |
| Beta | 1.00 | assumed | Damodaran industry betas: Machinery 0.96, Aerospace/Defense 0.95 |
| Debt / capital | 13% | assumed | Damodaran: Machinery 12.8%, Aerospace/Defense 13.5% |
| After-tax cost of debt | 4.0% | assumed | Damodaran, Machinery: 5.29% pre-tax, 3.97% after tax |
| WACC | 8.8% | computed | from the five above |
| Terminal growth | 2.5% | assumed | below the risk-free rate; under the ~3% that claims to outgrow the economy |
| Tax rate | 25% | assumed | 21% federal + state; Damodaran's own 5.29% → 3.97% implies the same 25% |
| ROIC | 20% | assumed | Damodaran after-tax ROIC: Machinery 24.4%, Aerospace/Defense 16.0%, Electrical Equipment 15.6% |
| Horizon | 5 years | assumed | FY2025–FY2029, as in Lectures 7–8 |

Damodaran figures are from his January 2026 industry tables. The valuation date is
February 2025; industry betas and capital structures move little in a year, but
strictly dated inputs would use his January 2025 tables.

**The fixed assumptions are the same for every forecast.** They set the level of every
valuation, but the only thing that differs between scenarios is year-1 growth and margin.
Part 5 checks whether the fixed assumptions could change which forecast wins.

In [3]:
# --- the discount rate, built the Lecture 7 way ---------------------------
RF, ERP, BETA = 0.045, 0.050, 1.00   # 10-yr UST Feb 2025; course ERP; industrial beta
KD_AT, D_WT = 0.040, 0.13            # after-tax cost of debt; debt / capital

ke = RF + BETA * ERP                 # cost of equity
WACC = round((1 - D_WT) * ke + D_WT * KD_AT, 3)

# --- everything else --------------------------------------------------------
G_TERM, YEARS = 0.025, 5   # perpetual growth; FY2025-FY2029 explicit
TAX, ROIC = 0.25, 0.20     # marginal tax; reinvestment = growth / ROIC

print(f"cost of equity {ke:.2%}  at {1 - D_WT:.0%} of capital")
print(f"cost of debt   {KD_AT:.2%}  at {D_WT:.0%} of capital (after tax)")
print(f"WACC           {WACC:.2%}")
print(f"\nimplied exit multiple of FCF: {(1 + G_TERM) / (WACC - G_TERM):.1f}x")

cost of equity 9.50%  at 87% of capital
cost of debt   4.00%  at 13% of capital (after tax)
WACC           8.80%

implied exit multiple of FCF: 16.3x


## Part 1 · The baseline 📏

The rule the language model has to beat: next year looks like the average of the
last three.

**Margin** is a true three-year average: FY2022, FY2023, FY2024.
**Growth** is the average of FY2023 and FY2024 growth. Three years of revenue give two
growth rates, and FY2021 is excluded on purpose (the data README keeps the window clear
of the pandemic rebound). It is a two-rate average, and it should be described as one.

A second benchmark, **last year**, uses FY2024 growth and FY2024 margin alone. It captures
momentum. If the model beats the three-year average but not last year, it may be reading
momentum rather than backlog.

In [4]:
def baseline(row):
    """Three-year-average forecast for one firm: (growth, margin)."""
    g23 = row["revenue_fy_minus_2"] / row["revenue_fy_minus_3"] - 1
    g24 = row["revenue_fy_minus_1"] / row["revenue_fy_minus_2"] - 1
    growth = (g23 + g24) / 2          # two growth rates in FY2022-FY2024
    margin = (row["margin_fy_minus_3"] + row["margin_fy_minus_2"]
              + row["margin_fy_minus_1"]) / 3
    return growth, margin


def last_year(row):
    """FY2024 alone: (growth, margin)."""
    growth = row["revenue_fy_minus_1"] / row["revenue_fy_minus_2"] - 1
    return growth, row["margin_fy_minus_1"]


rows = []
for tk, r in mx.iterrows():
    bg, bm = baseline(r)
    lg, lm = last_year(r)
    rows.append({"ticker": tk,
                 "base_growth": bg, "base_margin": bm,
                 "last_growth": lg, "last_margin": lm,
                 "actual_growth": r["actual_revenue_growth"],
                 "actual_margin": r["actual_margin"],
                 "backlog": bool(r["backlog_mentioned"]),
                 "flag": r["flag"]})
fc = pd.DataFrame(rows).set_index("ticker")

print(fc.drop(columns="flag").head(6).round(4).to_string())

        base_growth  base_margin  last_growth  last_margin  actual_growth  actual_margin  backlog
ticker                                                                                           
MMM         -0.0304      -0.0237      -0.0014       0.1962         0.0152         0.1855    False
ALLE         0.0745       0.1934       0.0333       0.2070         0.0782         0.2113    False
AME          0.0624       0.2531       0.0522       0.2564         0.0663         0.2581     True
AXON         0.3245       0.0690       0.3344       0.0281         0.3347        -0.0223    False
BA           0.0115      -0.0746      -0.1450      -0.1610         0.3450         0.0479     True
CARR         0.1414       0.1540       0.1865       0.1177        -0.0329         0.0999     True


Check one firm by hand before trusting the table.

In [5]:
r = mx.loc["CAT"]
g23 = r["revenue_fy_minus_2"] / r["revenue_fy_minus_3"] - 1
g24 = r["revenue_fy_minus_1"] / r["revenue_fy_minus_2"] - 1
m3 = [r["margin_fy_minus_3"], r["margin_fy_minus_2"], r["margin_fy_minus_1"]]

print(f"CAT revenue  {r['revenue_fy_minus_3']/1e9:.3f}  {r['revenue_fy_minus_2']/1e9:.3f}"
      f"  {r['revenue_fy_minus_1']/1e9:.3f}  ($bn, FY22-24)")
print(f"growth: ({g23:.4f} + {g24:.4f}) / 2 = {(g23 + g24) / 2:.4f}"
      f"   table {fc.loc['CAT', 'base_growth']:.4f}")
print(f"margin: mean of {[round(float(x), 4) for x in m3]} = {np.mean(m3):.4f}"
      f"   table {fc.loc['CAT', 'base_margin']:.4f}")

CAT revenue  59.427  67.060  64.809  ($bn, FY22-24)
growth: (0.1284 + -0.0336) / 2 = 0.0474   table 0.0474
margin: mean of [0.133, 0.1933, 0.2017] = 0.1760   table 0.1760


### How good is the rule the model has to beat?

Mean absolute error against what actually happened in FY2025. This is a preview of
Person 4's scoring, for the two benchmarks only.

In [6]:
GROUPS = {
    "all firms":  pd.Series(True, index=fc.index),
    "backlog":    fc["backlog"],
    "no backlog": ~fc["backlog"],
    "ex-flagged": fc["flag"] == "",
}


def mae(pred, actual):
    return (pred - actual).abs().mean()


acc = pd.DataFrame({
    name: {
        "n": int(mask.sum()),
        "growth MAE, 3-yr":  mae(fc.loc[mask, "base_growth"], fc.loc[mask, "actual_growth"]),
        "growth MAE, last":  mae(fc.loc[mask, "last_growth"], fc.loc[mask, "actual_growth"]),
        "growth bias, 3-yr": (fc.loc[mask, "base_growth"] - fc.loc[mask, "actual_growth"]).mean(),
        "margin MAE, 3-yr":  mae(fc.loc[mask, "base_margin"], fc.loc[mask, "actual_margin"]),
        "margin MAE, last":  mae(fc.loc[mask, "last_margin"], fc.loc[mask, "actual_margin"]),
    }
    for name, mask in GROUPS.items()
})
print(acc.to_string(float_format=lambda v: f"{v:.4f}"))

                   all firms  backlog  no backlog  ex-flagged
n                    53.0000  38.0000     15.0000     49.0000
growth MAE, 3-yr      0.0651   0.0747      0.0410      0.0542
growth MAE, last      0.0561   0.0659      0.0314      0.0498
growth bias, 3-yr     0.0108   0.0182     -0.0077      0.0280
margin MAE, 3-yr      0.0282   0.0242      0.0383      0.0236
margin MAE, last      0.0187   0.0195      0.0167      0.0154


Three things to take from that table:

- The three-year average misses **backlog firms by nearly twice as much** as the rest,
  and it overshoots them. That is the room a backlog-reading model has to work in — and
  the firms without backlog are a ready-made control group.
- **Last year beats the three-year average** on both growth and margin. The rubric's
  baseline is the three-year average, so it stays the official benchmark, but a result
  that beats only the weaker of the two should be reported as such.
- Some of the largest misses are acquisitions and divestitures, not backlog: XYL
  (Evoqua, FY23) and CARR (portfolio reshaping). A model that catches them from the
  MD&A is right for a reason that is not backlog.

In [7]:
fc["base_miss"] = fc["base_growth"] - fc["actual_growth"]
worst = fc.reindex(fc["base_miss"].abs().sort_values(ascending=False).index).head(8)
print(worst[["base_growth", "actual_growth", "base_miss", "backlog"]]
      .round(3).to_string())

fc.drop(columns="base_miss").to_csv(os.path.join(OUT, "baseline_forecasts.csv"))
print(f"\nsaved {OUT}/baseline_forecasts.csv  ({len(fc)} firms)")

        base_growth  actual_growth  base_miss  backlog
ticker                                                
BA            0.011          0.345     -0.333     True
DD           -0.238          0.019     -0.257     True
PCAR          0.089         -0.155      0.244     True
XYL           0.248          0.055      0.193     True
TEX          -0.049          0.131     -0.180     True
CARR          0.141         -0.033      0.174     True
OSK           0.139         -0.029      0.167     True
FTV          -0.143          0.019     -0.162    False

saved output/baseline_forecasts.csv  (53 firms)


## Part 2 · The DCF, tested on dummy numbers 🔧

### The engine from Lecture 6

In [8]:
def npv(rate, cashflows, first_at=1):
    """first_at=1: the first flow arrives one year from now."""
    total = 0.0
    for t, c in enumerate(cashflows):
        years = t + first_at
        total = total + c / (1 + rate) ** years
    return total


mine = npv(0.10, [100, 100, 100])
try:
    import numpy_financial as npf
    theirs = npf.npv(0.10, [0, 100, 100, 100])   # npf puts flows[0] at t = 0
    print(f"mine    : {mine:,.4f}")
    print(f"npf.npv : {theirs:,.4f}")
    print(f"agree   : {abs(mine - theirs) < 1e-9}")
except ImportError:
    print(f"mine: {mine:,.4f}   (numpy_financial not installed; expect 248.6852)")

mine: 248.6852   (numpy_financial not installed; expect 248.6852)


### Two decisions inside the DCF

**The forecast drives year 1, then fades.** Both the baseline and the model forecast
FY2025 only. Applied for five straight years, AXON's 32.5% baseline would compound into
a company four times its size, and every valuation gap would be that compounding.
Instead, year 1 uses the forecast and growth fades in a straight line to `G_TERM` by year 5.

**Growth costs money.** Reinvestment is `growth / ROIC` — the share of after-tax operating
profit that has to go back into the business to grow at that rate. A fixed reinvestment
rate would hand a higher growth forecast its extra value for free, which would flatter
whichever forecast says "faster". Reinvestment is floored at zero: a shrinking firm does
not get a cash windfall from shrinking.

In [9]:
def growth_path(g1, g_term=G_TERM, years=YEARS, fade=True):
    """Year 1 is the forecast; a straight line to g_term by the last year.

    fade=False holds the forecast for every explicit year (the alternative
    design, kept only so Part 4 can show what the fade is worth).
    """
    path = []
    for t in range(1, years + 1):
        w = (t - 1) / (years - 1) if fade else 0.0   # 0 in year 1
        path.append((1 - w) * g1 + w * g_term)
    return path


print("fade from 12%:", "  ".join(f"{g:.2%}" for g in growth_path(0.12)))
print("fade from -8%:", "  ".join(f"{g:.2%}" for g in growth_path(-0.08)))

fade from 12%: 12.00%  9.62%  7.25%  4.88%  2.50%
fade from -8%: -8.00%  -5.38%  -2.75%  -0.12%  2.50%


In [10]:
def dcf(rev0, growth, margin, wacc=WACC, g_term=G_TERM, tax=TAX,
        roic=ROIC, years=YEARS, fade=True, detail=False):
    """Enterprise value from FY2024 revenue, a year-1 growth and a margin."""
    if g_term >= wacc:
        raise ValueError(f"g_term {g_term:.2%} must be below WACC {wacc:.2%}")

    # 1. project revenue: the year-1 forecast, fading to g_term
    path = growth_path(growth, g_term, years, fade)
    rev, revs = rev0, []
    for g in path:
        rev = rev * (1 + g)
        revs.append(rev)

    # 2. free cash flow = after-tax operating profit less reinvestment
    fcf = []
    for g, r in zip(path, revs):
        nopat = r * margin * (1 - tax)
        reinvest = max(g, 0) / roic        # share of NOPAT put back to grow
        fcf.append(nopat * (1 - reinvest))

    # 3. discount each one back to the valuation date (early 2025)
    pv_e = npv(wacc, fcf)

    # 4. value everything after year 5, and discount that too
    fcf_next = revs[-1] * (1 + g_term) * margin * (1 - tax) * (1 - g_term / roic)
    tv = fcf_next / (wacc - g_term)
    pv_t = tv / (1 + wacc) ** years

    ev = pv_e + pv_t
    if detail:
        return {"growth": path, "revenue": revs, "fcf": fcf,
                "pv_explicit": pv_e, "pv_terminal": pv_t, "ev": ev}
    return ev

### Test 1 — a dummy firm, laid out year by year

Revenue of 1,000, 5% year-1 growth, a 15% margin. The table is built a second way —
vectorised with numpy, not the loop — and the two must agree.

In [11]:
REV, G1, M = 1000.0, 0.05, 0.15
d = dcf(REV, G1, M, detail=True)

t = np.arange(1, YEARS + 1)
g = np.linspace(G1, G_TERM, YEARS)                 # the same fade, vectorised
rev = REV * np.cumprod(1 + g)
nopat = rev * M * (1 - TAX)
fcf = nopat * (1 - np.maximum(g, 0) / ROIC)
disc = 1 / (1 + WACC) ** t
tv = rev[-1] * (1 + G_TERM) * M * (1 - TAX) * (1 - G_TERM / ROIC) / (WACC - G_TERM)
ev_check = (fcf * disc).sum() + tv * disc[-1]

table = pd.DataFrame({"growth": g, "revenue": rev, "NOPAT": nopat,
                      "reinvest %": np.maximum(g, 0) / ROIC, "FCF": fcf,
                      "discount": disc, "PV": fcf * disc},
                     index=pd.Index([f"FY{2024 + k}" for k in t], name="year"))
print(table.round(4).to_string())
print(f"\nPV of 5 years    : {d['pv_explicit']:>9,.2f}")
print(f"PV of terminal   : {d['pv_terminal']:>9,.2f}   ({d['pv_terminal'] / d['ev']:.1%} of EV)")
print(f"enterprise value : {d['ev']:>9,.2f}")
print(f"vectorised check : {ev_check:>9,.2f}   agree {abs(ev_check - d['ev']) < 1e-9}")

        growth    revenue     NOPAT  reinvest %       FCF  discount       PV
year                                                                        
FY2025  0.0500  1050.0000  118.1250      0.2500   88.5938    0.9191  81.4281
FY2026  0.0438  1095.9375  123.2930      0.2188   96.3226    0.8448  81.3712
FY2027  0.0375  1137.0352  127.9165      0.1875  103.9321    0.7764  80.6981
FY2028  0.0312  1172.5675  131.9138      0.1562  111.3023    0.7136  79.4307
FY2029  0.0250  1201.8817  135.2117      0.1250  118.3102    0.6559  77.6029

PV of 5 years    :    400.53
PV of terminal   :  1,262.59   (75.9% of EV)
enterprise value :  1,663.12
vectorised check :  1,663.12   agree True


### Test 2 — the identity

If year-1 growth already equals `G_TERM`, every year grows at `G_TERM` and the whole
DCF must collapse to the one-line Gordon formula: `FCF₁ / (WACC − g)`.

In [12]:
fcf1 = REV * (1 + G_TERM) * M * (1 - TAX) * (1 - G_TERM / ROIC)
gordon = fcf1 / (WACC - G_TERM)
model = dcf(REV, G_TERM, M)

print(f"Gordon, one line : {gordon:,.6f}")
print(f"dcf()            : {model:,.6f}")
print(f"difference       : {model - gordon:.1e}")

Gordon, one line : 1,601.562500
dcf()            : 1,601.562500
difference       : -9.1e-13


### Test 3 — the guards and the shape

In [13]:
# terminal growth at or above WACC must fail loudly, not return a negative number
try:
    dcf(REV, G1, M, g_term=0.09)
except ValueError as e:
    print("ValueError:", e)

# value is proportional to margin and to revenue: every cash flow carries both
print(f"\ndouble the margin  -> x{dcf(REV, G1, 2 * M) / dcf(REV, G1, M):.6f}")
print(f"double the revenue -> x{dcf(2 * REV, G1, M) / dcf(REV, G1, M):.6f}")

# shrinking: reinvestment floors at zero, so FCF equals NOPAT in a down year
s = dcf(REV, -0.10, M, detail=True)
yr1_nopat = s["revenue"][0] * M * (1 - TAX)
print(f"\n-10% year: FCF {s['fcf'][0]:,.2f}  NOPAT {yr1_nopat:,.2f}"
      f"  equal {abs(s['fcf'][0] - yr1_nopat) < 1e-9}")

# more growth is worth more here, because ROIC (20%) is above WACC (8.8%)
vals = [dcf(REV, x, M) for x in [-0.05, 0.0, 0.05, 0.10, 0.15]]
print(f"value rises with growth: {all(np.diff(vals) > 0)}")

ValueError: g_term 9.00% must be below WACC 8.80%

double the margin  -> x2.000000
double the revenue -> x2.000000

-10% year: FCF 101.25  NOPAT 101.25  equal True
value rises with growth: True


Because every cash flow is multiplied by the margin, **value is proportional to the
margin**. That gives an exact split of any valuation error into a growth part and a
margin part:

`(1 + total error) = (1 + growth part) × (1 + margin part)`

where the margin part is simply `forecast margin / actual margin − 1`. Part 3 uses this
to say which assumption drove the gap.

## Part 3 · The 53 firms 🏭

Every firm is valued once per forecast, at early 2025, from its FY2024 revenue. The
valuation that uses what **actually happened** in FY2025 is the yardstick; every other
forecast is scored by how far its value lands from it.

Two versions, as agreed:

- **Growth only** — every scenario uses the same margin, so only the growth forecast
  differs. This is the clean test of the backlog question. (Because value is proportional
  to margin, the percentage error does not depend on which margin is held fixed.)
- **Full** — each scenario uses its own growth *and* margin.

In [14]:
def value_all(growth, margin, **kw):
    """Enterprise value for every firm from one set of year-1 forecasts."""
    out = {}
    for tk in fc.index:
        out[tk] = dcf(mx.loc[tk, "revenue_fy_minus_1"], growth[tk], margin[tk], **kw)
    return pd.Series(out).rename_axis("ticker")


def score(growth, margin, **kw):
    """Valuation errors for one forecast, against the actual-results value."""
    true_full = value_all(fc["actual_growth"], fc["actual_margin"], **kw)
    hold = fc["base_margin"]                               # any fixed margin works
    g_only = value_all(growth, hold, **kw) / value_all(fc["actual_growth"], hold, **kw) - 1
    full = value_all(growth, margin, **kw) / true_full - 1
    m_part = margin / fc["actual_margin"] - 1
    return pd.DataFrame({"ev_true": true_full, "growth_only": g_only,
                         "full": full, "margin_part": m_part})


SCEN = {
    "3-yr average": (fc["base_growth"], fc["base_margin"]),
    "last year":    (fc["last_growth"], fc["last_margin"]),
}

### Where Person 3's forecasts plug in

Drop a file at `data/lm_forecasts.csv` with one row per firm per run:

| column | meaning |
| :--- | :--- |
| `ticker` | as in `master.csv` |
| `label` | e.g. `AI full`, `AI no MD&A` |
| `growth` | forecast FY2025 revenue growth, as a decimal (0.08 = 8%) |
| `margin` | forecast FY2025 operating margin, as a decimal |

Re-run the notebook and every table below includes those runs. Nothing else changes.

In [15]:
LM_FILE = os.path.join("..", "data", "lm_forecasts.csv")

if os.path.exists(LM_FILE):
    lm = pd.read_csv(LM_FILE)
    for label, part in lm.groupby("label"):
        part = part.set_index("ticker").reindex(fc.index)
        missing = part["growth"].isna().sum()
        if missing:
            print(f"{label}: {missing} firms missing -- they take the baseline")
        g = part["growth"].fillna(fc["base_growth"])
        m = part["margin"].fillna(fc["base_margin"])
        SCEN[label] = (g, m)
    print("loaded:", [k for k in SCEN])
else:
    print(f"no {LM_FILE} yet -- scoring the two benchmarks only")

no ..\data\lm_forecasts.csv yet -- scoring the two benchmarks only


In [16]:
res = {name: score(g, m) for name, (g, m) in SCEN.items()}

ev_true = res["3-yr average"]["ev_true"]
bad = ev_true[ev_true <= 0]
print(f"firms whose actual-results value is not positive: {len(bad)}  {list(bad.index)}")
print("  (a percentage error against a value <= 0 means nothing; they are left out of 'full')")

firms whose actual-results value is not positive: 1  ['AXON']
  (a percentage error against a value <= 0 means nothing; they are left out of 'full')


In [17]:
def summarise(res, mask):
    rows = {}
    for name, r in res.items():
        ok = mask & (r["ev_true"] > 0)
        rows[name] = {
            "n": int(mask.sum()),
            "growth only, mean |err|":   r.loc[mask, "growth_only"].abs().mean(),
            "growth only, median |err|": r.loc[mask, "growth_only"].abs().median(),
            "full, n": int(ok.sum()),
            "full, mean |err|":   r.loc[ok, "full"].abs().mean(),
            "full, median |err|": r.loc[ok, "full"].abs().median(),
        }
    return pd.DataFrame(rows)


for gname, mask in GROUPS.items():
    print(f"--- {gname} " + "-" * (60 - len(gname)))
    print(summarise(res, mask).to_string(float_format=lambda v: f"{v:.3f}"))
    print()

--- all firms ---------------------------------------------------
                           3-yr average  last year
n                                53.000     53.000
growth only, mean |err|           0.111      0.090
growth only, median |err|         0.053      0.044
full, n                          52.000     52.000
full, mean |err|                  0.256      0.200
full, median |err|                0.114      0.091

--- backlog -----------------------------------------------------
                           3-yr average  last year
n                                38.000     38.000
growth only, mean |err|           0.127      0.106
growth only, median |err|         0.059      0.076
full, n                          38.000     38.000
full, mean |err|                  0.270      0.219
full, median |err|                0.114      0.102

--- no backlog --------------------------------------------------
                           3-yr average  last year
n                                15

How to read it: "growth only, median |err| 0.053" means the three-year average's growth
forecast would have put the median firm's value 5.3% away from its actual-results value.
Means sit well above medians because a few firms (BA, PCAR, CARR, MMM) miss by 30% or more.
Report medians as the headline and means alongside.

The two benchmarks do not rank the same way everywhere: last year wins overall, but the
three-year average has the lower median among backlog firms and ex-flagged. "Which
benchmark is harder to beat" depends on the group, and the write-up should say so.

### Which assumption drove the DCF gap?

For each forecast, the full valuation error splits exactly into a growth part and a
margin part. The medians are the honest summary — a handful of firms with near-zero or
negative margins (BA, AXON, DD) produce percentage errors in the hundreds.

In [18]:
split = {}
for name, r in res.items():
    ok = (r["ev_true"] > 0) & (fc["flag"] == "")
    split[name] = {
        "n": int(ok.sum()),
        "median |total|":       r.loc[ok, "full"].abs().median(),
        "median |growth part|": r.loc[ok, "growth_only"].abs().median(),
        "median |margin part|": r.loc[ok, "margin_part"].abs().median(),
        "growth part bigger":   (r.loc[ok, "growth_only"].abs()
                                 > r.loc[ok, "margin_part"].abs()).mean(),
    }
print("ex-flagged, positive actual value\n")
print(pd.DataFrame(split).to_string(float_format=lambda v: f"{v:.3f}"))

# the identity behind the split, checked on every firm
r = res["3-yr average"]
ok = r["ev_true"] > 0
lhs = 1 + r.loc[ok, "full"]
rhs = (1 + r.loc[ok, "growth_only"]) * (1 + r.loc[ok, "margin_part"])
print(f"\n(1+total) = (1+growth)(1+margin) on every firm: {np.allclose(lhs, rhs)}")

ex-flagged, positive actual value

                      3-yr average  last year
n                           48.000     48.000
median |total|               0.108      0.091
median |growth part|         0.049      0.060
median |margin part|         0.115      0.063
growth part bigger           0.292      0.438

(1+total) = (1+growth)(1+margin) on every firm: True


**For the three-year average, the margin miss moves value more than the growth miss** — the
margin part is the larger of the two for about 70% of firms. There are two reasons. Value is
proportional to margin, so a 1.5-point miss on a 15% margin is a 10% value error before growth
enters at all. And the growth forecast only drives year 1 before it fades, so even a large
growth miss is diluted by four years of convergence and a terminal value that does not depend
on it.

That second reason is a modelling choice, so it is worth seeing what it is worth.

In [19]:
fade_cmp = {}
for label, kw in [("fade (base)", {}), ("no fade", {"fade": False})]:
    r = score(fc["base_growth"], fc["base_margin"], **kw)
    ok = (r["ev_true"] > 0) & (fc["flag"] == "")
    fade_cmp[label] = {
        "median |growth part|": r.loc[ok, "growth_only"].abs().median(),
        "median |margin part|": r.loc[ok, "margin_part"].abs().median(),
        "growth part bigger":   (r.loc[ok, "growth_only"].abs()
                                 > r.loc[ok, "margin_part"].abs()).mean(),
    }
print("3-yr average, ex-flagged\n")
print(pd.DataFrame(fade_cmp).to_string(float_format=lambda v: f"{v:.3f}"))

3-yr average, ex-flagged

                      fade (base)  no fade
median |growth part|        0.049    0.096
median |margin part|        0.115    0.115
growth part bigger          0.292    0.500


Holding the forecast for five years makes the growth part larger, as it must — but it does so
by assuming a one-year forecast is a five-year one, which is exactly the claim neither the
baseline nor the model makes. The fade stays. The memo should still say plainly that the
growth-versus-margin split depends on it.

In [20]:
r = res["3-yr average"].join(fc[["base_growth", "actual_growth",
                                 "base_margin", "actual_margin"]])
r = r[r["ev_true"] > 0]
show = r.reindex(r["full"].abs().sort_values(ascending=False).index).head(8)
print("largest 3-yr-average valuation errors\n")
print(show[["base_growth", "actual_growth", "base_margin", "actual_margin",
            "growth_only", "margin_part", "full"]].round(3).to_string())

largest 3-yr-average valuation errors

        base_growth  actual_growth  base_margin  actual_margin  growth_only  margin_part   full
ticker                                                                                         
BA            0.011          0.345       -0.075          0.048       -0.370       -2.559 -1.982
PCAR          0.089         -0.155        0.152          0.106        0.656        0.432  1.372
MMM          -0.030          0.015       -0.024          0.186       -0.083       -1.128 -1.117
CARR          0.141         -0.033        0.154          0.100        0.321        0.542  1.037
DE            0.005         -0.117        0.189          0.137        0.323        0.380  0.826
GNRC         -0.025         -0.020        0.115          0.069       -0.011        0.674  0.655
DD           -0.238          0.019        0.029          0.029       -0.465       -0.013 -0.472
VRT           0.187          0.277        0.112          0.179       -0.114       -0.372 -0.443


In [21]:
long = []
for name, r in res.items():
    g, m = SCEN[name]
    part = pd.DataFrame({"scenario": name, "growth": g, "margin": m,
                         "ev": value_all(g, m), "ev_actual": r["ev_true"],
                         "err_growth_only": r["growth_only"],
                         "err_margin_part": r["margin_part"],
                         "err_full": r["full"]})
    long.append(part)
long = pd.concat(long).rename_axis("ticker").reset_index()
long.to_csv(os.path.join(OUT, "dcf_valuations.csv"), index=False)
print(f"saved {OUT}/dcf_valuations.csv  ({len(long)} rows, {long['scenario'].nunique()} scenarios)")

saved output/dcf_valuations.csv  (106 rows, 2 scenarios)


## Part 4 · Sensitivity 🔲

### Growth by margin

Value per dollar of FY2024 revenue (an EV / revenue multiple), all fixed assumptions at
base. The axes span roughly the middle of the sample: baseline growth runs from about
−5% to +30%, margins from about 5% to 30%.

In [22]:
growths = [-0.05, 0.00, 0.05, 0.10, 0.15]
margins = [0.10, 0.125, 0.15, 0.175, 0.20]

grid = pd.DataFrame(
    [[dcf(1.0, g, m) for g in growths] for m in margins],
    index=[f"{m:.1%}" for m in margins],
    columns=[f"{g:.0%}" for g in growths])
grid.index.name = "margin"
grid.columns.name = "year-1 growth"
print("EV / FY2024 revenue\n")
print(grid.round(2).to_string())

print(f"\nmedian firm, 3-yr inputs: growth {fc['base_growth'].median():.1%},"
      f" margin {fc['base_margin'].median():.1%}"
      f" -> {dcf(1.0, fc['base_growth'].median(), fc['base_margin'].median()):.2f}x revenue")

EV / FY2024 revenue

year-1 growth   -5%    0%    5%   10%   15%
margin                                     
10.0%          0.92  1.03  1.11  1.19  1.28
12.5%          1.15  1.28  1.39  1.49  1.60
15.0%          1.38  1.54  1.66  1.79  1.92
17.5%          1.61  1.80  1.94  2.09  2.24
20.0%          1.84  2.06  2.22  2.39  2.56

median firm, 3-yr inputs: growth 6.7%, margin 15.2% -> 1.73x revenue


A level check, not a target: Damodaran's January 2026 EV/Sales for Machinery is 3.43x,
roughly twice what this model gives a median firm. The market is pricing more growth,
a longer runway or a lower discount rate than we assume. That does not matter for this
project — every scenario is scored as a *ratio* to the actual-results value, so the level
cancels — but it is why nobody should read these numbers as price targets.

### One assumption at a time — with the range next to the swing

A ranking by swing alone is a ranking of the ranges you chose (Lecture 8). So the table
reports the range each input was varied over, and the swing per percentage point.
Base case: the median firm's three-year inputs, per dollar of revenue.

In [23]:
G0, M0 = fc["base_growth"].median(), fc["base_margin"].median()
base_v = dcf(1.0, G0, M0)

TESTS = [   # (name, keyword or None, low, high)
    ("year-1 growth",   "growth", G0 - 0.05, G0 + 0.05),
    ("margin",          "margin", M0 - 0.03, M0 + 0.03),
    ("WACC",            "wacc",   WACC - 0.01, WACC + 0.01),
    ("terminal growth", "g_term", G_TERM - 0.01, G_TERM + 0.01),
    ("tax rate",        "tax",    0.20, 0.30),
    ("ROIC",            "roic",   0.15, 0.25),
]


def value_with(kw, x):
    args = {"growth": G0, "margin": M0}
    if kw in args:
        args[kw] = x
        return dcf(1.0, args["growth"], args["margin"])
    return dcf(1.0, G0, M0, **{kw: x})


rows = []
for name, kw, lo, hi in TESTS:
    v_lo, v_hi = value_with(kw, lo), value_with(kw, hi)
    width = (hi - lo) * 100
    rows.append({"input": name, "low": lo, "high": hi, "range (pp)": width,
                 "value low": v_lo, "value high": v_hi,
                 "swing": abs(v_hi - v_lo), "swing % of base": abs(v_hi - v_lo) / base_v,
                 "swing per pp": abs(v_hi - v_lo) / width})
tor = pd.DataFrame(rows).set_index("input").sort_values("swing", ascending=False)

print(f"base value {base_v:.3f}x revenue  (growth {G0:.1%}, margin {M0:.1%})\n")
print(tor.to_string(float_format=lambda v: f"{v:.3f}"))
print("\nranked by swing      :", list(tor.index))
print("ranked by swing / pp :", list(tor.sort_values("swing per pp", ascending=False).index))

base value 1.731x revenue  (growth 6.7%, margin 15.2%)

                  low  high  range (pp)  value low  value high  swing  swing % of base  swing per pp
input                                                                                               
margin          0.122 0.182       6.000      1.390       2.072  0.682            0.394         0.114
WACC            0.078 0.098       2.000      2.069       1.486  0.583            0.337         0.291
terminal growth 0.015 0.035       2.000      1.581       1.934  0.353            0.204         0.177
year-1 growth   0.017 0.117      10.000      1.605       1.861  0.256            0.148         0.026
tax rate        0.200 0.300      10.000      1.846       1.615  0.231            0.133         0.023
ROIC            0.150 0.250      10.000      1.627       1.793  0.166            0.096         0.017

ranked by swing      : ['margin', 'WACC', 'terminal growth', 'year-1 growth', 'tax rate', 'ROIC']
ranked by swing / pp : ['WACC', 'term

Ranked by swing, margin "matters most" — but it was varied over 6 points against WACC's 2.
Per percentage point, **WACC and terminal growth move the value most**, then margin, with
year-1 growth a distant fourth: one point of year-1 growth is worth about a quarter of one
point of margin, because it only drives a single year before fading.

That is the honest answer to "which assumption moves the value most": the discount rate and
terminal growth, per point. But they are identical for every forecast, so they cannot decide
the contest. The next table checks that.

### Can the fixed assumptions change who wins?

The question that matters for Project D is not how big a value is, but which forecast
lands closer to the actual-results value. So re-run the whole 53-firm scoring under each
alternative setting and watch the ranking.

In [24]:
ke_dam = RF + BETA * 0.0433                    # Damodaran implied ERP, Jan 2025
wacc_dam = round((1 - D_WT) * ke_dam + D_WT * KD_AT, 3)

SETTINGS = {
    "base":                     {},
    "WACC -1pp":                {"wacc": WACC - 0.01},
    "WACC +1pp":                {"wacc": WACC + 0.01},
    f"WACC {wacc_dam:.1%} (ERP 4.33%)": {"wacc": wacc_dam},
    "terminal growth 1.5%":     {"g_term": 0.015},
    "terminal growth 3.5%":     {"g_term": 0.035},
    "tax 21%":                  {"tax": 0.21},
    "ROIC 15%":                 {"roic": 0.15},
    "ROIC 25%":                 {"roic": 0.25},
    "no fade (hold 5 years)":   {"fade": False},
}

rows = {}
for sname, kw in SETTINGS.items():
    row = {}
    for name, (g, m) in SCEN.items():
        r = score(g, m, **kw)
        row[name] = r["growth_only"].abs().median()
    row["winner"] = min(SCEN, key=lambda k: row[k])
    rows[sname] = row
rob = pd.DataFrame(rows).T

print("median |valuation error|, growth only, all 53 firms\n")
print(rob.to_string(float_format=lambda v: f"{v:.4f}"))
print(f"\nwinner the same in every setting: {rob['winner'].nunique() == 1}")

median |valuation error|, growth only, all 53 firms

                       3-yr average last year     winner
base                         0.0533    0.0440  last year
WACC -1pp                    0.0580    0.0473  last year
WACC +1pp                    0.0486    0.0408  last year
WACC 8.2% (ERP 4.33%)        0.0561    0.0460  last year
terminal growth 1.5%         0.0505    0.0422  last year
terminal growth 3.5%         0.0564    0.0462  last year
tax 21%                      0.0533    0.0440  last year
ROIC 15%                     0.0417    0.0367  last year
ROIC 25%                     0.0594    0.0483  last year
no fade (hold 5 years)       0.1029    0.0850  last year

winner the same in every setting: True


The fixed assumptions move every error up or down together, so they change the size of the
gap, not who is closer. That is the property that lets one set of assumptions serve every
forecast. Re-run this table once the model's forecasts are in: if the winner flips under a
plausible setting, that is a finding to report, not a nuisance.

## Part 5 · For the team

**Person 3.** Write `data/lm_forecasts.csv` in the format in Part 3 and re-run. One label
per run (`AI full`, `AI no MD&A`). Record the model's training cutoff: FY2025 results were
published in early 2026, so a model trained past roughly January 2026 may already have seen
the answers.

**Person 4.** `output/baseline_forecasts.csv` has every forecast and the actuals, one row per
firm. `output/dcf_valuations.csv` has every valuation and error, one row per firm per
scenario. Score accuracy on FY2025 only; report "last year" alongside the three-year average;
split results by `backlog`; and report with and without the four flagged firms.